In [ ]:
"""
Environment Setup Module.
This cell installs the necessary quantum computing libraries in the Google Colab environment.
It must be executed before running any other cell.
"""
!pip install qiskit[visualization] qiskit-aer matplotlib pylatexenc

import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from IPython.display import display

print("\n ENVIRONMENT SETUP COMPLETE! You can now proceed to the next cell.")

### Exercise 3: Deutsch's Algorithm and the Quantum Oracle

**Motivation:**
Classically, if you have a black-box function $f(x)$ taking a 1-bit input, you must evaluate it twice ($x=0$ and $x=1$) to determine if the output is always the same (**Constant**) or divided 50/50 (**Balanced**). Deutsch's algorithm solves this in exactly 1 query by exploiting **Quantum Interference** and **Phase Kickback**.

**Theoretical Context:**
The algorithm uses an input qubit and an ancilla (workspace) qubit. By preparing the ancilla in the $|-\rangle$ state, the Oracle's output is not written to the target's probabilities, but instead "kicked back" as a phase shift on the input qubit. A final Hadamard gate translates this phase difference back into a measurable 0 or 1.

**Your Task:**
1. Run the code below exactly as provided. The Oracle section contains a `CNOT` gate, which acts as the balanced function $f(x) = x$. 
2. **Observe the result:** A 100% probability of measuring `1` proves the function is Balanced.
3. **The Experiment:** Go to the `STUDENT EXPERIMENT ZONE` in the code. Comment out the `qc.cx(0, 1)` line by typing a `#` in front of it. This removes the gate entirely, making the Oracle act as an Identity gate ($f(x) = 0$, which is a Constant function).
4. Run the code again. **Observe:** The interference perfectly flips, yielding a 100% probability of measuring `0`.

In [ ]:
"""
Deutsch's Algorithm Module.
Demonstrates quantum advantage by determining the global property 
(constant vs. balanced) of an oracle function in a single query.
"""

def run_deutsch_algorithm() -> None:
    """
    Builds, executes, and visualizes Deutsch's algorithm.
    The function evaluates an oracle and prints the logical conclusion.
    """
    # Initialize: 1 Input Qubit (q0), 1 Ancilla Qubit (q1), 1 Classical Bit
    qc = QuantumCircuit(2, 1)

    # --- 1. STATE PREPARATION ---
    # Prepare ancilla in |1> before applying the Hadamard gate
    qc.x(1)           
    
    # Apply Hadamard to both to create superposition and enable phase kickback
    qc.h(0)           # Input to |+>
    qc.h(1)           # Ancilla to |->
    qc.barrier()      # Visual and optimization barrier

    # --- 2. THE ORACLE (BLACK BOX) ---
    # ---------------------------------------------------------
    # 🧪 STUDENT EXPERIMENT ZONE: 
    # The CNOT gate below makes the oracle BALANCED (f(x) = x).
    # To make it CONSTANT (f(x) = 0), add a '#' at the start of the next line.
    
    qc.cx(0, 1)
    
    # ---------------------------------------------------------
    qc.barrier()

    # --- 3. INTERFERENCE & MEASUREMENT ---
    # Apply final Hadamard to the input qubit to extract the relative phase
    qc.h(0)           
    
    # Measure the input qubit (q0) into the classical bit (c0)
    qc.measure(0, 0)  

    # Draw the circuit diagram
    display(qc.draw('mpl'))

    # --- 4. EXECUTION ---
    # Run the circuit on the local simulator
    simulator = AerSimulator()
    job = simulator.run(qc, shots=1000)
    counts = job.result().get_counts()

    # --- 5. ANALYSIS ---
    print(f"Measurement Results: {counts}")
    
    # Interpret the results based on Deutsch's mathematical proof
    if '1' in counts and counts.get('1', 0) > 500:
        print(" Conclusion: The function is BALANCED (f(0) != f(1))")
    else:
        print(" Conclusion: The function is CONSTANT (f(0) == f(1))")
        
    display(plot_histogram(counts))

# Execute the algorithm
run_deutsch_algorithm()

### OPTIONAL: Execute on a Real Quantum Computer (QPU)

**The Reality of Quantum Hardware:**
Up to this point, we have used the `AerSimulator`. A simulator running on a classical computer executes mathematically perfect quantum logic. However, physical quantum computers (QPUs) operate at temperatures colder than deep space and manipulate delicate microwaves. They suffer from:
* **Decoherence:** The qubits lose their quantum state over time due to environmental interaction.
* **Gate Errors:** The microwave pulses applying the X, H, or CNOT gates are not 100% precise.



By running Deutsch's algorithm on a real IBM QPU, our perfect `100%` accuracy will degrade. We will measure the actual **Signal-to-Noise Ratio (SNR)** of a modern quantum chip.

**Step 1: Get Your API Token & Check Quota**
IBM provides a free "Open Plan" allowing 10 minutes of physical compute time per month. Note: This only counts the seconds the physical chip is actively running your circuit, not the time spent waiting in the queue.
1. Create a free account on the [IBM Quantum Platform](https://quantum.ibm.com/).
2. Copy your API token from the main dashboard.
3. Paste it into the `TOKEN` variable below and run the cell to check your remaining compute time.

In [ ]:
"""
IBM Quantum Authentication & Quota Check Module.
Authenticates the user and retrieves the remaining compute time on the real hardware.
"""
from qiskit_ibm_runtime import QiskitRuntimeService

# --- STUDENT INPUT REQUIRED ---
# Replace the string below with your personal IBM Quantum API Token
TOKEN = "YOUR_IBM_QUANTUM_TOKEN_HERE" 
# ------------------------------

def check_ibm_quota(api_token: str) -> None:
    """
    Authenticates with IBM Quantum and prints the current usage limits.
    
    Args:
        api_token (str): The IBM Quantum API token.
    """
    if api_token == "YOUR_IBM_QUANTUM_TOKEN_HERE":
        print(" ACTION REQUIRED: Please insert your personal API token in the code above.")
        return

    print("Authenticating with IBM Quantum...")
    try:
        # Save account locally for this session
        QiskitRuntimeService.save_account(channel="ibm_quantum", token=api_token, set_as_default=True, overwrite=True)
        service = QiskitRuntimeService()
        
        # Retrieve usage statistics
        usage = service.usage()
        print("\n Authentication Successful!")
        print("=== Your IBM Quantum Compute Time ===")
        print(f"Usage Details: {usage}")
        print("=====================================")
        print("You can proceed to the next cell to execute your circuit on the QPU.")
        
    except Exception as e:
        print(f" Authentication Failed. Error: {e}")

# Run the quota check
check_ibm_quota(TOKEN)

**Step 2: Transpile and Execute**
Different quantum computers have different physical wiring (Topologies). Some qubits can talk to each other directly, others cannot. 
The code below will:
1. Find the least busy quantum computer available to you.
2. **Transpile** (translate) our logical circuit into the physical microwave instructions specific to that chip.
3. Send the job to the queue and wait for the results.

*(Warning: Depending on global traffic, your job might stay in the queue for a few minutes or even an hour. Leave the notebook open while it processes).*

In [ ]:
"""
QPU Execution Module.
Transpiles the circuit for the target hardware, executes it using the modern SamplerV2 primitive,
and analyzes the impact of quantum noise on the result.
"""
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2
from qiskit.visualization import plot_histogram
from IPython.display import display

def run_on_real_hardware() -> None:
    """
    Finds the least busy backend, transpiles the previously built circuit, 
    executes it, and calculates the accuracy against environmental noise.
    """
    # Verify authentication from the previous cell
    try:
        service = QiskitRuntimeService()
    except Exception:
        print(" Please run the authentication cell above first.")
        return

    # 1. Select the least busy real hardware (excluding simulators)
    print("Searching for the least busy quantum computer...")
    backend = service.least_busy(operational=True, simulator=False)
    print(f" Selected Backend: {backend.name} (Qubits: {backend.num_qubits})")

    # 2. Transpile the circuit
    # 'qc' is the QuantumCircuit built in Exercise 3.
    # We use optimization_level=1 for basic gate routing.
    print("Transpiling logical circuit to physical hardware instructions...")
    pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
    isa_circuit = pm.run(qc)

    # 3. Execute using the SamplerV2 primitive
    print(f"Sending job to the queue of {backend.name}... (Please wait)")
    sampler = SamplerV2(backend)
    job = sampler.run([isa_circuit])
    print(f"Job successfully submitted! Job ID: {job.job_id()}")
    
    # 4. Wait for completion and extract results
    result = job.result()
    pub_result = result[0]
    
    # Extract the dictionary of measurement counts
    # Note: Modern primitives nest the classical register data. 
    # 'c' is the default name of the classical register.
    counts = pub_result.data.c.get_counts()
    
    print(f"\n Real Hardware Results Received: {counts}")
    
    # 5. Noise Analysis
    # Since we used a CNOT oracle (Balanced), the correct theoretical answer is '1'
    ideal_answer = '1' 
    correct_shots = counts.get(ideal_answer, 0)
    total_shots = sum(counts.values())
    accuracy = (correct_shots / total_shots) * 100
    
    print("\n=== NOISE ANALYSIS ===")
    print(f"Ideal Theoretical Output: 100% '{ideal_answer}'")
    print(f"Actual Hardware Accuracy: {accuracy:.2f}%")
    error_rate = 100 - accuracy
    print(f"Error Rate (Decoherence/Gate Noise): {error_rate:.2f}%")
    print("======================")
    
    # Plot the noisy results
    display(plot_histogram(counts, title=f"Real QPU Execution on {backend.name}"))

# IMPORTANT: Ensure the 'qc' variable from Exercise 3 is still in memory.
# Execute the hardware job
if TOKEN != "YOUR_IBM_QUANTUM_TOKEN_HERE":
    run_on_real_hardware()